# Dual-Arm Cohort Exploration (HPC ↔️ Colab)

**Purpose:** Mirror the scripted analytics in `ehr/cohort_eda.py` inside a notebook workflow so rapid iteration on Colab remains aligned with the HPC pipeline.

*Project focus:* 30-day readmission risk for MIMIC-IV inpatients, targeting two high-impact cohorts:
- Short-stay transitional cases (LOS ≤7 d)
- Cardio-renal/sepsis long stays (LOS ≥15 d with AKI/HF/sepsis clusters)


## 0. Environment & Data Contracts

- PhysioNet paths mounted read-only (on Colab: use `physionet` Google Drive mirror or upload subsets).
- Local intermediates: `data/interim/cohort.csv`, derived CSVs in `data/interim/readmit_analysis/`.
- Run-time toggles (`SKIP_LABS`, `SKIP_MEDS`) mimic flags in `ehr/cohort_eda.py` for long-running steps.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path('data') / 'interim'
READMIT_DIR = DATA_ROOT / 'readmit_analysis'
PHYSIONET_ROOT = Path('physionet.org') / 'files' / 'mimiciv' / '3.1' / 'hosp'

COHORT_PATH = DATA_ROOT / 'cohort.csv'
DIAG_PATH = PHYSIONET_ROOT / 'diagnoses_icd.csv.gz'
LABEVENTS_PATH = PHYSIONET_ROOT / 'labevents.csv.gz'
PRESCRIPTIONS_PATH = PHYSIONET_ROOT / 'prescriptions.csv.gz'

TODAY = pd.Timestamp.today().strftime('%Y-%m-%d')
SKIP_LABS = True   # toggle to False when running on HPC
SKIP_MEDS = True   # toggle to False when running on HPC


## 1. Load Cohort & Recreate Filters
Replicates the scripted cohort loader with LOS segmentation.


In [ ]:
cohort_cols = [
    'subject_id','hadm_id','admittime','dischtime','length_of_stay_days',
    'readmitted_within_window','readmission_gap_in_days','age_at_admit',
    'gender','race','admission_type','discharge_location','discharge_note_text'
]
cohort = pd.read_csv(
    COHORT_PATH, usecols=cohort_cols, parse_dates=['admittime','dischtime']
)

LOS_BINS = [0,4,7,10,14,21,float('inf')]
LOS_LABELS = ['<=4d','5-7d','8-10d','11-14d','15-21d','>=22d']
cohort['los_segment'] = pd.cut(
    cohort['length_of_stay_days'], bins=LOS_BINS, labels=LOS_LABELS, include_lowest=True
)
cohort.head()


## 2. ICD-Based Condition Flags
Mirror cardio-renal/sepsis tagging used for the dual focus cohorts.


In [ ]:
condition_prefixes = {
    'acute_kidney_injury': ['N17','N19','N185','584','586'],
    'heart_failure': ['I50','I13','I11','428'],
    'hyponatremia': ['E871','2761'],
    'posthemorrhagic_anemia': ['D62','2851'],
    'sepsis': ['A41','R652','R651','9959','038']
}

hadm_set = set(cohort['hadm_id'])
condition_hits = {name: set() for name in condition_prefixes}

for chunk in pd.read_csv(DIAG_PATH, chunksize=500_000, usecols=['hadm_id','icd_code']):
    chunk = chunk[chunk['hadm_id'].isin(hadm_set)]
    if chunk.empty:
        continue
    codes = chunk['icd_code'].astype(str).str.upper().str.replace('.', '', regex=False)
    for name, prefixes in condition_prefixes.items():
        mask = np.zeros(len(codes), dtype=bool)
        for prefix in prefixes:
            mask |= codes.str.startswith(prefix).to_numpy()
        if mask.any():
            condition_hits[name].update(chunk.loc[mask, 'hadm_id'])

for name, hadms in condition_hits.items():
    cohort[f'has_{name}'] = cohort['hadm_id'].isin(hadms)

cohort['has_any_cardiorenal_sepsis'] = cohort[[
    'has_acute_kidney_injury','has_heart_failure','has_hyponatremia',
    'has_posthemorrhagic_anemia','has_sepsis'
]].any(axis=1)
cohort['has_aki_and_hf'] = cohort['has_acute_kidney_injury'] & cohort['has_heart_failure']
cohort['cardiorenal_sepsis_long'] = (cohort['length_of_stay_days'] >= 15) & cohort['has_any_cardiorenal_sepsis']
cohort['long_stay_no_cardiorenal'] = (cohort['length_of_stay_days'] >= 15) & (~cohort['has_any_cardiorenal_sepsis'])


## 3. LOS & Focus Group Summaries
Reproduce tables cited in the 2025-11-01 daily report.


In [ ]:
los_summary = (
    cohort.groupby('los_segment')
    .agg(admissions=('hadm_id','count'),
         readmits=('readmitted_within_window','sum'),
         readmit_rate=('readmitted_within_window','mean'),
         cardiorenal_share=('cardiorenal_sepsis_long','mean'))
    .reset_index()
)
los_summary['readmit_rate_pct'] = (los_summary['readmit_rate']*100).round(1)
los_summary['cardiorenal_share_pct'] = (los_summary['cardiorenal_share']*100).round(1)
los_summary


In [ ]:
cluster_masks = {
    'cardiorenal_long': cohort['cardiorenal_sepsis_long'],
    'long_no_cardiorenal': cohort['long_stay_no_cardiorenal'],
    'aki_hf_combo': cohort['has_aki_and_hf'],
    'sepsis_any': cohort['has_sepsis']
}
cluster_rows = []
for name, mask in cluster_masks.items():
    subset = cohort[mask]
    cluster_rows.append({
        'cluster': name,
        'admissions': int(mask.sum()),
        'readmits': int(subset['readmitted_within_window'].sum()),
        'readmit_rate_pct': round(subset['readmitted_within_window'].mean()*100, 1) if len(subset) else np.nan
    })
cluster_summary = pd.DataFrame(cluster_rows)
cluster_summary


> Clinical perspective: Short stays (≤7 d) account for ~15% readmit rate across >120 k admissions, while cardio-renal long stays exceed 25%. Target discharge planning and renal stabilization accordingly.


## 4. Discharge Disposition & Follow-up Documentation
Assess transition-of-care risk drivers.


In [ ]:
cohort['disposition_clean'] = cohort['discharge_location'].fillna('UNKNOWN').str.lower().str.strip()
disp_summary = (
    cohort.groupby('disposition_clean')
    .agg(admissions=('hadm_id','count'),
         readmits=('readmitted_within_window','sum'),
         readmit_rate=('readmitted_within_window','mean'))
    .reset_index()
)
disp_summary = disp_summary[disp_summary['admissions'] >= 100].copy()
disp_summary['readmit_rate_pct'] = (disp_summary['readmit_rate']*100).round(1)
disp_summary.sort_values('readmit_rate', ascending=False).head(10)


In [ ]:
keywords = ['follow-up','follow up','clinic appointment','pcp','primary care','see physician','appointment','call your doctor']
text_lower = cohort['discharge_note_text'].fillna('').str.lower()
follow_mask = np.zeros(len(text_lower), dtype=bool)
for kw in keywords:
    follow_mask |= text_lower.str.contains(kw).to_numpy()
cohort['has_followup_note'] = follow_mask
follow_summary = (
    cohort.groupby('has_followup_note')
    .agg(admissions=('hadm_id','count'),
         readmits=('readmitted_within_window','sum'),
         readmit_rate=('readmitted_within_window','mean'),
         median_gap=('readmission_gap_in_days','median'))
    .reset_index()
)
follow_summary['readmit_rate_pct'] = (follow_summary['readmit_rate']*100).round(1)
follow_summary


## 5. Time-to-Readmit Distribution
Highlight early bounce-backs for each cohort.


In [ ]:
positives = cohort[cohort['readmitted_within_window'] == 1].copy()
positives['readmission_gap_in_days'] = positives['readmission_gap_in_days'].clip(lower=0, upper=30)
positives['gap_bucket'] = pd.cut(positives['readmission_gap_in_days'], bins=[0,7,14,21,30], labels=['0-7d','8-14d','15-21d','22-30d'], include_lowest=True)

def share(df, groups):
    out = df.groupby(groups).size().reset_index(name='readmit_count')
    totals = out.groupby(groups[:-1])['readmit_count'].transform('sum')
    out['share_pct'] = (out['readmit_count']/totals*100).round(1)
    return out

los_gap = share(positives, ['los_segment','gap_bucket'])
los_gap.head(12)


In [ ]:
cardio_gap = share(positives, ['cardiorenal_sepsis_long','gap_bucket'])
cardio_gap


## 6. Optional: Labs & Medication Exposure
Set `SKIP_LABS` / `SKIP_MEDS` to `False` when running on HPC; these steps can exceed Colab memory/time limits.


In [ ]:
if not SKIP_LABS:
    long_ids = set(cohort[cohort['length_of_stay_days'] >= 15]['hadm_id'])
    discharge_map = cohort.set_index('hadm_id')['dischtime'].to_dict()
    item_map = {'creatinine':[50912,52546], 'sodium':[50983,52623]}
    lab_records = []
    for chunk in pd.read_csv(LABEVENTS_PATH, chunksize=1_000_000, usecols=['hadm_id','itemid','charttime','valuenum']):
        chunk = chunk[chunk['hadm_id'].isin(long_ids)]
        if chunk.empty:
            continue
        chunk = chunk[chunk['itemid'].isin(sum(item_map.values(), []))]
        chunk['charttime'] = pd.to_datetime(chunk['charttime'])
        chunk['dischtime'] = chunk['hadm_id'].map(discharge_map)
        chunk = chunk.dropna(subset=['dischtime'])
        chunk['hours_before_discharge'] = (chunk['dischtime'] - chunk['charttime']).dt.total_seconds()/3600
        chunk = chunk[(chunk['hours_before_discharge'] >= 0) & (chunk['hours_before_discharge'] <= 48)]
        for analyte, ids in item_map.items():
            sub = chunk[chunk['itemid'].isin(ids)].sort_values(['hadm_id','hours_before_discharge'])
            latest = sub.groupby('hadm_id').first().reset_index()
            latest['analyte'] = analyte
            lab_records.append(latest[['hadm_id','analyte','valuenum','hours_before_discharge']])
    if lab_records:
        labs = pd.concat(lab_records)
        labs = labs.merge(cohort[['hadm_id','cardiorenal_sepsis_long','has_aki_and_hf']], on='hadm_id', how='left')
        lab_summary = labs.groupby(['analyte','cardiorenal_sepsis_long']).valuenum.median().reset_index()
        display(lab_summary)
    else:
        print('No lab records found in final 48 hours window.')
else:
    print('SKIP_LABS=True — skipping heavy lab extraction.')


In [ ]:
if not SKIP_MEDS:
    long_ids = set(cohort[cohort['length_of_stay_days'] >= 15]['hadm_id'])
    discharge_map = cohort.set_index('hadm_id')['dischtime'].to_dict()
    drug_map = {
        'loop_diuretics': ['FUROSEMIDE','BUMETANIDE','TORSEMIDE'],
        'k_sparing_or_thiazide': ['SPIRONOLACTONE','CHLORTHALIDONE','HYDROCHLOROTHIAZIDE'],
        'ace_arb': ['LISINOPRIL','LOSARTAN','VALSARTAN','CAPTOPRIL','ENALAPRIL']
    }
    exposure = {label: set() for label in drug_map}
    for chunk in pd.read_csv(PRESCRIPTIONS_PATH, chunksize=500_000, usecols=['hadm_id','drug','starttime','stoptime']):
        chunk = chunk[chunk['hadm_id'].isin(long_ids)]
        if chunk.empty:
            continue
        chunk['drug'] = chunk['drug'].astype(str).str.upper()
        chunk['starttime'] = pd.to_datetime(chunk['starttime'], errors='coerce')
        chunk['stoptime'] = pd.to_datetime(chunk['stoptime'], errors='coerce')
        chunk['dischtime'] = chunk['hadm_id'].map(discharge_map)
        for label, kws in drug_map.items():
            mask = np.zeros(len(chunk), dtype=bool)
            for kw in kws:
                mask |= chunk['drug'].str.contains(kw).to_numpy()
            sub = chunk[mask]
            if sub.empty:
                continue
            window_start = sub['dischtime'] - pd.Timedelta(hours=48)
            overlap = sub[
                ((sub['starttime'] <= sub['dischtime']) & (sub['starttime'] >= window_start)) |
                ((sub['stoptime'] >= window_start) & (sub['stoptime'] <= sub['dischtime'])) |
                ((sub['starttime'] <= window_start) & (sub['stoptime'] >= sub['dischtime']))
            ]
            exposure[label].update(overlap['hadm_id'].unique())
    pd.DataFrame({k: [len(v)] for k, v in exposure.items()})
else:
    print('SKIP_MEDS=True — skipping prescriptions scan.')


## 7. Prior Utilization & Demographic Slices


In [ ]:
cohort = cohort.sort_values(['subject_id','admittime'])
cohort['prior_admissions'] = cohort.groupby('subject_id').cumcount()
cohort['prior_admissions_bucket'] = pd.cut(
    cohort['prior_admissions'], bins=[-1,0,1,2,10], labels=['0','1','2','3+']
)
util_summary = (
    cohort.groupby('prior_admissions_bucket')
    .agg(admissions=('hadm_id','count'),
         readmits=('readmitted_within_window','sum'),
         readmit_rate=('readmitted_within_window','mean'),
         median_age=('age_at_admit','median'))
    .reset_index()
)
util_summary['readmit_rate_pct'] = (util_summary['readmit_rate']*100).round(1)
util_summary


## 8. Social & Documentation Signals
Quick heuristics for social determinants captured in discharge notes.


In [ ]:
indicators = {
    'lives_alone': ['lives alone','living alone','resides alone'],
    'transport_barrier': ['transportation','no ride','no transport','lack of transportation'],
    'housing_insecure': ['homeless','no fixed address','shelter'],
    'language_barrier': ['interpreter','language barrier','translated by'],
    'social_support': ['family support','caregiver','daughter will','son will']
}
for key, phrases in indicators.items():
    mask = np.zeros(len(text_lower), dtype=bool)
    for phrase in phrases:
        mask |= text_lower.str.contains(phrase).to_numpy()
    cohort[key] = mask

social_summary = []
for key in indicators:
    subset = cohort[cohort[key]]
    social_summary.append({
        'indicator': key,
        'admissions': len(subset),
        'readmit_rate_pct': round(subset['readmitted_within_window'].mean()*100, 1) if len(subset) else np.nan,
        'short_los_share_pct': round(subset['los_segment'].isin(['<=4d','5-7d']).mean()*100, 1) if len(subset) else np.nan
    })

pd.DataFrame(social_summary)


## 9. Outputs & Handoff
- Optionally write refreshed tables to `data/interim/readmit_analysis/` with today's stamp (disabled by default).
- Document clinical insights here before updating daily report templates.
- For HPC parity, promote changes back into `ehr/cohort_eda.py` and rerun batch jobs.
